## Import Libraries

In [3]:
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait as WDW
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

## Setup and Configure Selenium WebDriver

In [4]:
print("Setting up Webdriver....")
chrome_opt = Options() # Initialize the chrome webdriver
chrome_opt.add_argument("--headless")
chrome_opt.add_argument("--disable-gpu")
chrome_opt.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.6778.265 Safari/537.36")
print("configuration done!")

Setting up Webdriver....
configuration done!


In [5]:
# Setting up webdriver: Installation
print("Installing Chrome webdriver")
service = Service(ChromeDriverManager ().install())
print("Final setup")
driver = webdriver.Chrome(service=service, options=chrome_opt)

Installing Chrome webdriver
Final setup


## Making a connection to the webpage

In [7]:
URL = "https://www.glasses.com/gl-us/eyeglasses"

In [8]:
print(f"Visiting {URL} page")
driver.get(URL)

# Further Instructions
try:
    print("Waiting for product tiles to load")
    WDW(driver, 10).until(EC.presence_of_element_located((By.CLASS_NAME, 'catalog-page')))
    print("Done, Proceed!")
except TimeoutError as e:
    print(f"Expected tag did not load on time: {e}")

Visiting https://www.glasses.com/gl-us/eyeglasses page
Waiting for product tiles to load
Done, Proceed!


In [10]:
content = driver.page_source
page = BeautifulSoup(content, "html.parser")

## Data Extraction

In [ ]:
product_tiles = page.find_all("a", class_="product-tile")
print(len(product_tiles))
print(f"found {len(product_tiles)} products")

26


In [17]:
products = []

for tile in product_tiles:
    product_info = tile.find('div', class_='product-info')

    if product_info:
        name_tag = product_info.find('div', class_='product-code')
        name = name_tag.text if name_tag else "Unknown"

        # brand
        brand_tag = product_info.find('div', class_='product-brand')
        brand = brand_tag.text if brand_tag else "Unknown"
        
        # price
        price_container = product_info.find('div', class_='product-prices-container')
        if price_container:
            # former price
            former_price_tag = price_container.find('div', class_='product-list-price')
            former_price = former_price_tag.text if former_price_tag else 'Unknown'
            # Current price
            current_price_tag = price_container.find('div', class_='product-offer-price')
            current_price = current_price_tag.text if current_price_tag else 'Unknown'
        else:
            former_price = current_price = "Unknown"
    else:
        brand = name = former_price = current_price = "Unknown"

    data = {
        "Product_Name": name,
        "Brand": brand,
        "Former_Price": former_price,
        "Current_Price": current_price
    }
    print(data)
    products.append(data)

{'Product_Name': 'DG3404', 'Brand': 'Dolce & Gabbana', 'Former_Price': '$ 422.00', 'Current_Price': '$ 295.40'}
{'Product_Name': 'MK3068 Portland', 'Brand': 'Michael Kors', 'Former_Price': '$ 175.00', 'Current_Price': '$ 87.50'}
{'Product_Name': 'PH2283U', 'Brand': 'Polo Ralph Lauren', 'Former_Price': '$ 237.00', 'Current_Price': '$ 165.90'}
{'Product_Name': 'VE1302', 'Brand': 'Versace', 'Former_Price': '$ 526.00', 'Current_Price': '$ 368.20'}
{'Product_Name': 'RB7021 Matthew', 'Brand': 'Ray-Ban', 'Former_Price': '$ 176.00', 'Current_Price': '$ 88.00'}
{'Product_Name': 'VO5615', 'Brand': 'Vogue Eyewear', 'Former_Price': '$ 128.00', 'Current_Price': '$ 89.60'}
{'Product_Name': 'JC3025', 'Brand': 'Jimmy Choo', 'Former_Price': '$ 287.00', 'Current_Price': '$ 200.90'}
{'Product_Name': 'FZ8020U', 'Brand': 'Scuderia Ferrari', 'Former_Price': '$ 191.00', 'Current_Price': '$ 133.70'}
{'Product_Name': 'PH2251U', 'Brand': 'Polo Ralph Lauren', 'Former_Price': '$ 183.00', 'Current_Price': '$ 91.50

In [18]:
# Step 3 - Data Storage: store the extracted data in CSV and JSON formats
import csv
import json
# Save to CSV file
column_name = products[0].keys() # get the column names
with open('glassesdotcom.csv', mode='w', newline='', encoding='utf-8') as csv_file: # open up the file with context manager
    dict_writer = csv.DictWriter(csv_file, fieldnames=column_name)
    dict_writer.writeheader()
    dict_writer.writerows(products)
print(f"Saved {len(products)} records to CSV in the extracted data folder.")

# Save to JSON file
with open("glassesdotcom.json", mode='w') as json_file:
    json.dump(products, json_file, indent=4)
print(f"Saved {len(products)} records to JSON in the extracted data folder.")

# close the browser
driver.quit()
print("End of Web Extraction")

Saved 26 records to CSV in the extracted data folder.
Saved 26 records to JSON in the extracted data folder.
End of Web Extraction
